In [51]:
import pandas as pd
import numpy as np


In [52]:
df= pd.read_csv("/content/questionAnswer.csv")

In [53]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [54]:
# tokenize
def tokenize(text: str)-> list:
  text= text.replace('?' ,"")
  text= text.replace("'", "")
  return text.lower().split(" ")

In [55]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [56]:
# makes a dictionary of vocabulary
vocab= {'<UNK>': 0}   #unk means undefined/unknown here
i=1

In [57]:
for row in df.values:
  question, answer= row
  tokenized_question= tokenize(question)
  tokenized_answer= tokenize(answer)
  tokens= tokenized_question + tokenized_answer
  for token in tokens:
    if token not in vocab:
      vocab[token]= i
      i= i+1


In [58]:
len(vocab)

324

In [59]:
# convert word into numerical indices
def to_indices(text):
  indices= []

  for token in tokenize(text):
    if token in vocab:
      indices.append(vocab.get(token, ""))

    else:
      indices.append(vocab.get( "<UNK>"))

  return indices

In [60]:
to_indices(df.iloc[0, 0])

[1, 2, 3, 4, 5, 6]

In [61]:
df['question']= df['question'].apply(to_indices)
df['answer']= df['answer'].apply(to_indices)

In [62]:
df.sample(10)

,question,answer
61,"[1, 87, 229, 230, 231, 232]",[233]
51,"[1, 2, 3, 146, 86, 19, 192, 193]",[194]
64,"[10, 96, 3, 104, 239]",[240]
81,"[78, 79, 288, 81, 19, 14, 289]",[85]
73,"[42, 137, 2, 138, 39, 175, 269]",[99]
18,"[78, 79, 80, 81, 82, 83, 84]",[85]
82,"[42, 290, 291, 118, 292, 158, 293, 294]",[295]
19,"[42, 86, 87, 88, 89, 39, 90]",[91]
59,"[1, 2, 3, 221, 5, 222, 223, 224]",[225]
89,"[42, 137, 2, 62, 39, 3, 322, 323]",[6]


In [63]:
df.values.ndim

2

In [64]:
df.iloc[1].values

array([list([1, 2, 3, 4, 5, 8]), list([9])], dtype=object)

In [65]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [66]:
class CustomDataset(Dataset):
  def __init__(self, df):
    self.df= df


  def __len__(self):
    return df.shape[0]



  def __getitem__(self, index):
    return torch.tensor(self.df.iloc[index]['question']),torch.tensor(self.df.iloc[index]['answer'][0])


In [67]:
dataset= CustomDataset(df)

In [68]:
len(dataset)

90

In [69]:
dataloader= DataLoader(dataset, pin_memory= True, shuffle= True, batch_size= 1)

In [70]:
for x, y in dataloader:
  print(x, y)
  print("-"*50)

tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([155])
--------------------------------------------------
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([6])
--------------------------------------------------
tensor([[ 42, 216, 118, 217, 218,  19,  14, 219,  43]]) tensor([220])
--------------------------------------------------
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
--------------------------------------------------
tensor([[ 10,  96,   3, 104, 239]]) tensor([240])
--------------------------------------------------
tensor([[ 42, 200,   2,  14, 201, 202, 203, 204]]) tensor([205])
--------------------------------------------------
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([321])
--------------------------------------------------
tensor([[  1,   2,   3, 180, 181, 182, 183]]) tensor([184])
--------------------------------------------------
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([254])
--------------------------------------------------
te

In [71]:
# make RNN layer
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):

    super().__init__()

    self.embedding= nn.Embedding(num_embeddings= vocab_size, embedding_dim= 50)  #return in shape(Batch_size, no. wof words, dimension of embedding)
    self.rnn= nn.RNN(50, 64, nonlinearity='relu', batch_first= True)

    self.fc= nn.Linear(64, vocab_size)


  def forward(self, question):
    embedded_words= self.embedding(question)
    hidden_state, rnn_output= self.rnn(embedded_words)   #rnn gives 2 output; hidden state value at each time sequence and final output of the rnn layer

    output= self.fc(rnn_output.squeeze(0))
    return output




In [72]:
model= SimpleRNN(len(vocab)).to(device= 'cuda')

In [73]:
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(params= model.parameters(), lr= 0.001)

In [74]:
model.train()

SimpleRNN(
  (embedding): Embedding(324, 50)
  (rnn): RNN(50, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=324, bias=True)
)

In [75]:
for i in range(20):
  total_loss= 0
  for question, answer in dataloader:
    optimizer.zero_grad()

    question= question.to(device= 'cuda')
    answer= answer.to(device= 'cuda')



    #calculate y_pred:
    y_pred= model(question)


    loss= criterion(y_pred, answer)
    total_loss= total_loss + loss.item()

    loss.backward()

    optimizer.step()


  print(f"Epoch: {i+ 1}; loss: {total_loss/len(dataloader)}")









Epoch: 1; loss: 5.797303252749973
Epoch: 2; loss: 4.509453645017412
Epoch: 3; loss: 2.772737330198288
Epoch: 4; loss: 1.3492082662052578
Epoch: 5; loss: 0.5930416595604685
Epoch: 6; loss: 0.32647496021042266
Epoch: 7; loss: 0.20392248024129206
Epoch: 8; loss: 0.12491026495893796
Epoch: 9; loss: 0.09482251058410232
Epoch: 10; loss: 0.08092703345335192
Epoch: 11; loss: 0.04474882300969006
Epoch: 12; loss: 0.04535622232748816
Epoch: 13; loss: 0.04003797470876533
Epoch: 14; loss: 0.02742209736542362
Epoch: 15; loss: 0.012041165891827809
Epoch: 16; loss: 0.009808160858746204
Epoch: 17; loss: 0.008216726104728877
Epoch: 18; loss: 0.006883621383329026
Epoch: 19; loss: 0.0060295898350887
Epoch: 20; loss: 0.0053676432936400585


In [76]:
model.eval()

SimpleRNN(
  (embedding): Embedding(324, 50)
  (rnn): RNN(50, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=324, bias=True)
)

In [84]:
def prediction(Question: str):


  indexed_question= to_indices(Question)
  indexed_question= torch.tensor(indexed_question).unsqueeze(0).to(device= 'cuda')
  with torch.no_grad():
    y_hat= model(indexed_question)
  prob= nn.functional.softmax(y_hat, dim= 1)
  _ ,idx= torch.max(prob, dim= 1)
  if idx< 0.1:
    print("I dont know")

  else:
    keys= list(vocab.keys())
    print(keys[idx])


In [92]:
prediction("capital of france")

paris
